<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/xGBoostModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Load the sparse dataset
import pickle
with open("sparse_dict.pkl", "rb") as f:
  sparse = pickle.load(f)


In [3]:
print(sparse.keys())

dict_keys(['sparse_train', 'sparse_val', 'sparse_test'])


In [7]:
for name, hts in sparse.items():
  print(name, hts.shape)

sparse_train (1271440, 56)
sparse_val (423814, 56)
sparse_test (423814, 56)


Prepare train, validation and test sets

In [8]:
#Separate features and target variable
X_train = sparse["sparse_train"].drop(columns=["FinalTestResult"])
y_train = sparse["sparse_train"]["FinalTestResult"]

X_val = sparse["sparse_val"].drop(columns=["FinalTestResult"])
y_val = sparse["sparse_val"]["FinalTestResult"]

X_test = sparse["sparse_test"].drop(columns=["FinalTestResult"])
y_test = sparse["sparse_test"]["FinalTestResult"]

Encode categorical variables

In [10]:
#xgboost requires numerical outputs
import pandas as pd
from sklearn.preprocessing import LabelEncoder

X_train_enc = X_train.copy()
X_val_enc = X_val.copy()
X_test_enc = X_test.copy()

for col in X_train_enc.columns:
    if X_train_enc[col].dtype == "object":
        le = LabelEncoder()

        combined = pd.concat([
            X_train_enc[col],
            X_val_enc[col],
            X_test_enc[col]
        ]).astype(str)

        le.fit(combined)

        X_train_enc[col] = le.transform(X_train_enc[col].astype(str))
        X_val_enc[col] = le.transform(X_val_enc[col].astype(str))
        X_test_enc[col] = le.transform(X_test_enc[col].astype(str))

Encode target variable

In [11]:
#Check the output, if not in numericals, encode it
print(y_train.value_counts())

FinalTestResult
NEGATIVE    1239327
POSITIVE      32113
Name: count, dtype: int64


In [12]:
#Encode target variables in all the target sets

target_encoder = LabelEncoder()

y_train_enc = target_encoder.fit_transform(y_train)
y_val_enc = target_encoder.transform(y_val)
y_test_enc = target_encoder.transform(y_test)

In [19]:
y_train.value_counts(normalize=True)

,proportion
FinalTestResult,
NEGATIVE,0.974743
POSITIVE,0.025257


In [20]:
Calculate this if positivity is much fewer than negatives: This is because HIV positivity is usually imbalanced
neg = (y_train_enc == 0).sum()
pos = (y_train_enc == 1).sum()

scale_pos_weight = neg / pos

print(scale_pos_weight)

38.592688319372215


Train XGBoost

In [13]:
pip install xgboost

In [21]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=2231,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train_enc,
    y_train_enc
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

Validate the model

In [22]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

val_pred = xgb_model.predict(X_val_enc)
val_prob = xgb_model.predict_proba(X_val_enc)[:,1]

In [23]:
#Check Accuracy
print("Validation Accuracy:",
      accuracy_score(y_val_enc, val_pred))

Validation Accuracy: 0.7923994960053231


In [24]:
#Check AUC
print("Validation AUC:",
      roc_auc_score(y_val_enc, val_prob))

Validation AUC: 0.8675413385836388


In [25]:
print(classification_report(y_val_enc, val_pred))

              precision    recall  f1-score   support

           0       0.99      0.79      0.88    413110
           1       0.09      0.77      0.16     10704

    accuracy                           0.79    423814
   macro avg       0.54      0.78      0.52    423814
weighted avg       0.97      0.79      0.86    423814



The model as is has a recall of 77%. Predicts 77% of people to turn positive but only 9% of the people turns positive leading to so many false postives

In [26]:
#Check ROC-AUC
from sklearn.metrics import roc_auc_score

val_prob = xgb_model.predict_proba(X_val_enc)[:,1]

roc_auc_score(y_val_enc, val_prob)

np.float64(0.8675413385836388)

In [27]:
#Check PR-AUC
from sklearn.metrics import average_precision_score

average_precision_score(y_val_enc, val_prob)

np.float64(0.25388613997138554)

In [28]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val_enc, val_pred)
print(cm)

[[327588  85522]
 [  2462   8242]]


Retrain the model

In [29]:
#Retrain with validation monitoring

xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=2231,

    n_estimators=2000,
    learning_rate=0.03,

    max_depth=4,
    min_child_weight=5,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=1,
    reg_lambda=5
)

xgb_model.fit(
    X_train_enc,
    y_train_enc,
    eval_set=[(X_val_enc, y_val_enc)],
    verbose=100
)

[0]	validation_0-aucpr:0.08578
[100]	validation_0-aucpr:0.18374
[200]	validation_0-aucpr:0.20579
[300]	validation_0-aucpr:0.21985
[400]	validation_0-aucpr:0.22917
[500]	validation_0-aucpr:0.23729
[600]	validation_0-aucpr:0.24269
[700]	validation_0-aucpr:0.24788
[800]	validation_0-aucpr:0.25267
[900]	validation_0-aucpr:0.25589
[1000]	validation_0-aucpr:0.25957
[1100]	validation_0-aucpr:0.26287
[1200]	validation_0-aucpr:0.26547
[1300]	validation_0-aucpr:0.26836
[1400]	validation_0-aucpr:0.27098
[1500]	validation_0-aucpr:0.27364
[1600]	validation_0-aucpr:0.27585
[1700]	validation_0-aucpr:0.27807
[1800]	validation_0-aucpr:0.27998
[1900]	validation_0-aucpr:0.28182
[1999]	validation_0-aucpr:0.28361


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=2000,
              n_jobs=None, num_parallel_tree=None, ...)

In [30]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

val_prob = xgb_model.predict_proba(X_val_enc)[:,1]

print("ROC-AUC:",
      roc_auc_score(y_val_enc, val_prob))

print("PR-AUC:",
      average_precision_score(y_val_enc, val_prob))

ROC-AUC: 0.8721729128269401
PR-AUC: 0.28364486674294365


In [31]:
val_pred = (val_prob >= 0.5).astype(int)

print(classification_report(y_val_enc, val_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    413110
           1       0.72      0.07      0.13     10704

    accuracy                           0.98    423814
   macro avg       0.85      0.54      0.56    423814
weighted avg       0.97      0.98      0.97    423814



In [32]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]

for t in thresholds:

    pred = (val_prob >= t).astype(int)

    print(
        f"Threshold={t:.2f}",
        f"Precision={precision_score(y_val_enc,pred):.3f}",
        f"Recall={recall_score(y_val_enc,pred):.3f}",
        f"F1={f1_score(y_val_enc,pred):.3f}"
    )

Threshold=0.05 Precision=0.143 Recall=0.629 F1=0.233
Threshold=0.10 Precision=0.237 Recall=0.447 F1=0.310
Threshold=0.15 Precision=0.316 Recall=0.338 F1=0.327
Threshold=0.20 Precision=0.392 Recall=0.262 F1=0.314
Threshold=0.25 Precision=0.459 Recall=0.208 F1=0.286
Threshold=0.30 Precision=0.522 Recall=0.168 F1=0.254
Threshold=0.40 Precision=0.624 Recall=0.112 F1=0.190
Threshold=0.50 Precision=0.717 Recall=0.073 F1=0.132
